# LeetCode 1756 — Design Most Recently Used Queue bằng Python

Link đề: https://leetcode.com/problems/design-most-recently-used-queue/ · Độ khó: **Medium** · Chủ đề: Array, Linked List, Design, Binary Indexed Tree, Ordered Set

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class MRUQueue:

    def __init__(self, n: int):
        ...

    def fetch(self, k: int) -> int:
        ...


# Your MRUQueue object will be instantiated and called as such:
# obj = MRUQueue(n)
# param_1 = obj.fetch(k)
```

## 1. Mục tiêu học

1. Đọc và **mô phỏng** một bài thiết kế (design) theo định dạng LeetCode: danh sách lệnh + danh sách tham số → danh sách kết quả.
2. Cài bản cơ bản bằng `list` (`pop(k−1)` + `append`) và hiểu vì sao mỗi `fetch` tốn `O(n)`.
3. Giải **follow-up** bằng **chia căn** (sqrt decomposition) — `O(√N)` mỗi lệnh.
4. Giải follow-up bằng **cây Fenwick (BIT)** + "tìm phần tử thứ k" bằng nhảy nhị phân — `O(log N)` mỗi lệnh.
5. Kiểm thử các cài đặt bằng chuỗi lệnh ngẫu nhiên có seed cố định, đối chiếu với bản mô phỏng đơn giản nhất.

## 2. Đề bài & ví dụ

Thiết kế cấu trúc giống hàng đợi, đưa phần tử **vừa được dùng** về **cuối** hàng đợi:
- `MRUQueue(n)`: khởi tạo hàng đợi `[1, 2, 3, …, n]`.
- `fetch(k)`: chuyển phần tử thứ `k` (**đánh số từ 1**) về cuối hàng đợi và **trả về** nó.

**Ví dụ 1:**
```
Input:  ["MRUQueue", "fetch", "fetch", "fetch", "fetch"]
        [[8], [3], [5], [2], [8]]
Output: [null, 3, 6, 2, 2]
```
- Khởi tạo `[1,2,3,4,5,6,7,8]`.
- `fetch(3)`: phần tử thứ 3 là `3` → `[1,2,4,5,6,7,8,3]`, trả `3`.
- `fetch(5)`: phần tử thứ 5 là `6` → `[1,2,4,5,7,8,3,6]`, trả `6`.
- `fetch(2)`: phần tử thứ 2 là `2` → `[1,4,5,7,8,3,6,2]`, trả `2`.
- `fetch(8)`: phần tử thứ 8 là `2` (đã ở cuối) → giữ nguyên, trả `2`.

**Ràng buộc:** `1 ≤ n ≤ 2000`, `1 ≤ k ≤ n`, tối đa `2000` lần gọi `fetch`.

**Follow-up:** `O(n)` mỗi `fetch` khá dễ. Bạn có thể làm tốt hơn không?

In [1]:
import random
import time
import math

def run(cls, ops, args):
    """Bộ chạy kiểu LeetCode: ops[0] là tên lớp, các ops sau là tên phương thức."""
    out, obj = [], None
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

ex_ops = ["MRUQueue", "fetch", "fetch", "fetch", "fetch"]
ex_args = [[8], [3], [5], [2], [8]]
ex_out = [None, 3, 6, 2, 2]
print(ex_ops, ex_args, "-> kỳ vọng", ex_out)

['MRUQueue', 'fetch', 'fetch', 'fetch', 'fetch'] [[8], [3], [5], [2], [8]] -> kỳ vọng [None, 3, 6, 2, 2]


## 3. Kiến thức nền

### 3.1 `list.pop(i)` và `list.append(x)`

`pop(i)` xóa và trả phần tử ở chỉ số `i`, phải **dịch** mọi phần tử phía sau sang trái ⇒ `O(n − i)`. `append` thêm vào cuối, `O(1)` khấu hao.

In [2]:
q = [1, 2, 3, 4, 5]
x = q.pop(2)          # chỉ số 0-based 2 = phần tử thứ 3
q.append(x)
print(x, q)

3 [1, 2, 4, 5, 3]


### 3.2 Chỉ số 1-based và 0-based

Đề đánh số từ 1: phần tử thứ `k` nằm ở chỉ số Python `k − 1`. Đây là nguồn lỗi lệch một rất phổ biến.

In [3]:
q = [10, 20, 30]
k = 1
print("phần tử thứ", k, "là", q[k - 1], "(không phải q[k] =", q[k], ")")

phần tử thứ 1 là 10 (không phải q[k] = 20 )


### 3.3 Lớp (class) và bộ chạy lệnh

Bài design yêu cầu viết một **lớp** với hàm khởi tạo `__init__` và các phương thức. Hàm `run` ở trên mô phỏng cách LeetCode gọi: tạo đối tượng với tham số đầu, rồi gọi lần lượt từng phương thức; kết quả của hàm khởi tạo ghi là `None` (tương ứng `null`).

### 3.4 Cây Fenwick (Binary Indexed Tree) — tổng tiền tố và cập nhật `O(log N)`

Mảng `tree` 1-based; `add(i, v)` cộng `v` tại vị trí `i`; `prefix(i)` = tổng `a[1..i]`. Mẹo `i & -i` lấy bit thấp nhất.

In [4]:
class Fenwick:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def add(self, i, v):
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

fw = Fenwick(8)
for i in [1, 2, 3, 5, 8]:
    fw.add(i, 1)
print([fw.prefix(i) for i in range(9)], "| 12 & -12 =", 12 & -12)

[0, 1, 2, 3, 3, 4, 4, 4, 5] | 12 & -12 = 4


### 3.5 Tìm vị trí thứ `k` trong BIT bằng nhảy nhị phân

Nếu `a[i] ∈ {0, 1}` (1 = ô đang có phần tử), vị trí nhỏ nhất có `prefix(pos) ≥ k` tìm được trong `O(log N)` bằng cách thử các bước nhảy `2^j` giảm dần.

In [5]:
def kth(fw, k):
    pos, step = 0, 1 << fw.n.bit_length()
    while step:
        nxt = pos + step
        if nxt <= fw.n and fw.t[nxt] < k:
            pos = nxt; k -= fw.t[nxt]
        step >>= 1
    return pos + 1

print([kth(fw, k) for k in range(1, 6)], "(các ô đang bật: 1,2,3,5,8)")

[1, 2, 3, 5, 8] (các ô đang bật: 1,2,3,5,8)


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay Ví dụ 1 chỉ cần **thao tác trên dãy**: lấy phần tử thứ `k`, xóa nó, gắn vào cuối. Điều khó là làm **nhanh**:
- Xóa ở giữa một mảng ⇒ dịch phần sau ⇒ `O(n)`.
- Danh sách liên kết xóa `O(1)` nhưng phải **đi bộ** tới phần tử thứ `k` ⇒ `O(n)`.

Hai thao tác cần nhanh: (1) **tìm phần tử thứ k**, (2) **xóa** nó và **thêm vào cuối**.

Quan sát then chốt: phần tử bị `fetch` luôn được gắn vào **cuối**. Nếu ta cấp cho mỗi phần tử một **"ô"** (slot) theo thời gian — ban đầu phần tử `i` ở ô `i`, mỗi lần `fetch` thứ `t` gắn phần tử vào ô mới `n + t` — thì thứ tự trong hàng đợi chính là **thứ tự tăng dần của các ô đang có người**. Tìm phần tử thứ `k` = tìm ô có người thứ `k` ⇒ bài toán "đếm / tìm thứ k" kinh điển của BIT.

In [6]:
n = 8
slot_owner = {i: i for i in range(1, n + 1)}
print("ban đầu (ô -> phần tử):", slot_owner)
# fetch(3): phần tử ở ô có người thứ 3 là ô 3 (giá trị 3) -> chuyển sang ô 9
v = slot_owner.pop(3); slot_owner[9] = v
print("sau fetch(3):", [slot_owner[s] for s in sorted(slot_owner)])

ban đầu (ô -> phần tử): {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8}
sau fetch(3): [1, 2, 4, 5, 6, 7, 8, 3]


## 5. Cách 1 — Brute Force / baseline: dựng lại list mỗi lần

**Ý tưởng:** làm đúng như mô tả — cắt list thành ba phần rồi ghép lại: `q[:k-1] + q[k:] + [q[k-1]]`.

In [7]:
class MRUQueueBrute:
    def __init__(self, n: int):
        self.q = list(range(1, n + 1))
    def fetch(self, k: int) -> int:
        x = self.q[k - 1]
        self.q = self.q[:k - 1] + self.q[k:] + [x]       # tạo list mới mỗi lần
        return x

got = run(MRUQueueBrute, ex_ops, ex_args)
print(got); assert got == ex_out

[None, 3, 6, 2, 2]


**Độ phức tạp:** mỗi `fetch` tạo list mới `O(n)` ⇒ `2000 · 2000 = 4·10⁶` thao tác — vẫn qua giới hạn đề, nhưng tốn bộ nhớ cấp phát liên tục.

**Nút thắt:** cả "tìm thứ k" lẫn "xóa" đều tuyến tính. Bước đầu: sửa tại chỗ bằng `pop` + `append` (vẫn `O(n)` nhưng nhanh hơn nhiều nhờ dịch bộ nhớ bằng C).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chia căn):** chia hàng đợi thành các **khối** độ dài ~`B = √N`. Tìm phần tử thứ `k`: đi qua các khối, trừ dần độ dài ⇒ `O(N/B)`; xóa trong một khối `O(B)`; thêm vào khối cuối `O(1)`. Chọn `B ≈ √N` ⇒ `O(√N)`. Khi khối cuối quá to hoặc có quá nhiều khối rỗng thì **dựng lại**.

**Insight 2 (ô theo thời gian + BIT):** tổng số ô cần dùng `≤ n + số lần fetch ≤ 4000`.
- `bit[s] = 1` nếu ô `s` đang có phần tử.
- `fetch(k)`: tìm ô `s` = vị trí thứ `k` có `bit = 1` (nhảy nhị phân), lấy giá trị `val[s]`, tắt ô `s`, bật ô mới `n + t`, gán `val[n + t] = val[s]`.
- Mỗi thao tác `O(log N)`.

**Invariant:** thứ tự hàng đợi = thứ tự tăng dần của các ô đang bật; số ô bật luôn bằng `n`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `list.pop(k − 1)` + `append` (tại chỗ)

In [8]:
class MRUQueueList:
    def __init__(self, n: int):
        self.q = list(range(1, n + 1))
    def fetch(self, k: int) -> int:
        x = self.q.pop(k - 1)
        self.q.append(x)
        return x

print(run(MRUQueueList, ex_ops, ex_args))

[None, 3, 6, 2, 2]


**Code nộp LeetCode (Cách 2):** chính là lớp trên (đổi tên thành `MRUQueue`). **Độ phức tạp:** `fetch` `O(n)`, khởi tạo `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — Chia căn (sqrt decomposition)

**Giải thích từng đoạn:**
1. Khởi tạo: chia `[1..n]` thành các khối độ dài `B = max(1, ⌈√n⌉)`.
2. `fetch(k)`: duyệt các khối, nếu `k > len(khối)` thì trừ đi và sang khối sau; ngược lại phần tử là `khối[k − 1]`, `pop` nó.
3. Thêm phần tử vào khối cuối (tạo khối mới nếu khối cuối đã đầy `B`).
4. Khi số khối vượt `2·⌈n/B⌉ + 2`, dựng lại toàn bộ (khử khối rỗng) — xảy ra hiếm, chi phí `O(n)` chia đều.

In [9]:
class MRUQueueSqrt:
    def __init__(self, n: int):
        self.n = n
        self.B = max(1, math.isqrt(n) + (math.isqrt(n) ** 2 < n))
        self._build(list(range(1, n + 1)))

    def _build(self, items):
        B = self.B
        self.blocks = [items[i:i + B] for i in range(0, len(items), B)] or [[]]

    def fetch(self, k: int) -> int:
        for bi, blk in enumerate(self.blocks):
            if k > len(blk):
                k -= len(blk)
                continue
            x = blk.pop(k - 1)
            break
        last = self.blocks[-1]
        if len(last) >= self.B:
            self.blocks.append([x])
        else:
            last.append(x)
        if len(self.blocks) > 2 * (self.n // self.B + 1) + 2:       # quá nhiều khối (nhiều khối rỗng/thưa)
            self._build([v for b in self.blocks for v in b])
        return x

print(run(MRUQueueSqrt, ex_ops, ex_args))

[None, 3, 6, 2, 2]


**Code nộp LeetCode (Cách 3):** đổi tên lớp thành `MRUQueue`. **Độ phức tạp:** `fetch` `O(√n)` khấu hao (duyệt `O(n/B)` khối + `pop` `O(B)` + dựng lại hiếm khi), Space `O(n)`.

### 7.3 Cách 4 — Cây Fenwick theo "ô thời gian" + tìm thứ k

In [10]:
class MRUQueueBIT:
    MAX_FETCH = 2000

    def __init__(self, n: int):
        self.n = n
        self.size = n + self.MAX_FETCH
        self.tree = [0] * (self.size + 1)
        self.val = [0] * (self.size + 1)
        for i in range(1, self.size + 1):           # dựng BIT O(N): đẩy lên cha
            if i <= n:
                self.val[i] = i
                self.tree[i] += 1
            j = i + (i & -i)
            if j <= self.size:
                self.tree[j] += self.tree[i]
        self.t = 0                                  # số lần fetch đã làm

    def _add(self, i, v):
        while i <= self.size:
            self.tree[i] += v
            i += i & -i

    def _kth(self, k):
        pos, step = 0, 1 << self.size.bit_length()
        while step:
            nxt = pos + step
            if nxt <= self.size and self.tree[nxt] < k:
                pos = nxt
                k -= self.tree[nxt]
            step >>= 1
        return pos + 1

    def fetch(self, k: int) -> int:
        s = self._kth(k)
        x = self.val[s]
        self.t += 1
        new = self.n + self.t
        self._add(s, -1)
        self._add(new, 1)
        self.val[new] = x
        return x

print(run(MRUQueueBIT, ex_ops, ex_args))

[None, 3, 6, 2, 2]


Lưu ý: vòng dựng BIT trong `__init__` đi `i` tăng dần qua **toàn bộ** `1..size` (không chỉ `1..n`): khi tới `i`, `tree[i]` đã nhận đủ đóng góp từ các con nên đẩy tiếp lên cha `i + (i & -i)` — dựng trong `O(N)` thay vì `n` lần `add` (`O(n log N)`). Nếu chỉ chạy tới `n`, các nút `> n` sẽ không truyền tổng lên cha và BIT bị sai.

**Code nộp LeetCode (Cách 4):** xem mục 9. **Độ phức tạp:** khởi tạo `O(N)`, `fetch` `O(log N)`, Space `O(N)` với `N = n + 2000`.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 (`n = 8`): in ô được tìm thấy, giá trị, ô mới, và hàng đợi hiện tại (đọc theo thứ tự ô đang bật).

| fetch | k | ô thứ k | giá trị | ô mới | hàng đợi sau |
|---|---|---|---|---|---|
| 1 | 3 | 3 | 3 | 9 | 1 2 4 5 6 7 8 3 |
| 2 | 5 | 6 | 6 | 10 | 1 2 4 5 7 8 3 6 |
| 3 | 2 | 2 | 2 | 11 | 1 4 5 7 8 3 6 2 |
| 4 | 8 | 11 | 2 | 12 | 1 4 5 7 8 3 6 2 |

In [11]:
class TraceBIT(MRUQueueBIT):
    def queue(self):
        return [self.val[s] for s in range(1, self.size + 1)
                if self._prefix(s) - self._prefix(s - 1) == 1]
    def _prefix(self, i):
        r = 0
        while i > 0:
            r += self.tree[i]; i -= i & -i
        return r
    def fetch(self, k):
        s = self._kth(k)
        x = super().fetch(k)
        print(f"fetch({k}): ô thứ {k} là ô {s}, giá trị {x}, chuyển sang ô {self.n + self.t} | hàng đợi: {self.queue()}")
        return x

assert run(TraceBIT, ex_ops, ex_args) == ex_out

fetch(3): ô thứ 3 là ô 3, giá trị 3, chuyển sang ô 9 | hàng đợi: [1, 2, 4, 5, 6, 7, 8, 3]
fetch(5): ô thứ 5 là ô 6, giá trị 6, chuyển sang ô 10 | hàng đợi: [1, 2, 4, 5, 7, 8, 3, 6]
fetch(2): ô thứ 2 là ô 2, giá trị 2, chuyển sang ô 11 | hàng đợi: [1, 4, 5, 7, 8, 3, 6, 2]
fetch(8): ô thứ 8 là ô 11, giá trị 2, chuyển sang ô 12 | hàng đợi: [1, 4, 5, 7, 8, 3, 6, 2]


Trace Cách 3 (chia căn) — in các khối sau mỗi lệnh:

In [12]:
class TraceSqrt(MRUQueueSqrt):
    def fetch(self, k):
        x = super().fetch(k)
        print(f"fetch({k}) -> {x} | B={self.B}, khối: {self.blocks}")
        return x

assert run(TraceSqrt, ex_ops, ex_args) == ex_out

fetch(3) -> 3 | B=3, khối: [[1, 2], [4, 5, 6], [7, 8, 3]]
fetch(5) -> 6 | B=3, khối: [[1, 2], [4, 5], [7, 8, 3], [6]]
fetch(2) -> 2 | B=3, khối: [[1], [4, 5], [7, 8, 3], [6, 2]]
fetch(8) -> 2 | B=3, khối: [[1], [4, 5], [7, 8, 3], [6, 2]]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** (BIT, `O(log N)` mỗi `fetch`) — đáp ứng follow-up. (Nếu chỉ cần qua giới hạn đề, Cách 2 với 3 dòng là đủ.)

In [13]:
class MRUQueue:
    def __init__(self, n: int):
        self.n = n
        self.size = n + 2000                       # tối đa 2000 lần fetch
        self.tree = [0] * (self.size + 1)
        self.val = [0] * (self.size + 1)
        for i in range(1, self.size + 1):          # dựng BIT trong O(N)
            if i <= n:
                self.val[i] = i
                self.tree[i] += 1
            j = i + (i & -i)
            if j <= self.size:
                self.tree[j] += self.tree[i]
        self.used = n

    def _add(self, i: int, v: int) -> None:
        while i <= self.size:
            self.tree[i] += v
            i += i & -i

    def fetch(self, k: int) -> int:
        pos, step = 0, 1 << self.size.bit_length()
        while step:
            nxt = pos + step
            if nxt <= self.size and self.tree[nxt] < k:
                pos = nxt
                k -= self.tree[nxt]
            step >>= 1
        s = pos + 1
        x = self.val[s]
        self._add(s, -1)
        self.used += 1
        self._add(self.used, 1)
        self.val[self.used] = x
        return x


# Your MRUQueue object will be instantiated and called as such:
# obj = MRUQueue(n)
# param_1 = obj.fetch(k)
got = run(MRUQueue, ex_ops, ex_args)
print(got); assert got == ex_out

[None, 3, 6, 2, 2]


## 10. Vì sao thuật toán đúng

**Cách 2** đúng hiển nhiên: `pop(k−1)` lấy đúng phần tử thứ `k` (1-based) và `append` đưa nó về cuối.

**Cách 3:** invariant — nối các khối theo thứ tự cho ra đúng hàng đợi. Việc duyệt trừ dần độ dài khối tìm đúng khối chứa phần tử thứ `k` và vị trí trong khối; `pop` + thêm vào khối cuối giữ invariant; dựng lại chỉ gom các khối theo đúng thứ tự.

**Cách 4:** invariant — (I1) tập ô bật có đúng `n` phần tử; (I2) đọc `val` của các ô bật theo thứ tự ô tăng dần cho ra đúng hàng đợi; (I3) `tree` là BIT của mảng bật/tắt.
- Khởi tạo: ô `1..n` bật với `val[i] = i` ⇒ (I1–I3) đúng.
- `fetch(k)`: nhảy nhị phân tìm ô nhỏ nhất có tổng tiền tố `= k`, tức ô bật thứ `k` ⇒ theo (I2) là phần tử thứ `k`. Tắt ô đó và bật ô mới `used + 1` — lớn hơn mọi ô đã dùng ⇒ phần tử nằm **cuối** thứ tự ⇒ (I2) giữ. Số ô bật không đổi ⇒ (I1). `_add` cập nhật BIT ⇒ (I3). ∎

Ô mới không bao giờ vượt `size` vì có tối đa 2000 lần `fetch`.

## 11. Phân tích Time / Space Complexity

Gọi `n` = kích thước hàng đợi, `Q` = số lần `fetch` (`≤ 2000`), `N = n + Q`.

| Cách | `__init__` | `fetch` | Space |
|---|---|---|---|
| 1. Dựng lại list | `O(n)` | `O(n)` (cấp phát mới) | `O(n)` |
| 2. `pop` + `append` | `O(n)` | `O(n)` (dịch bằng C) | `O(n)` |
| 3. Chia căn | `O(n)` | `O(√n)` khấu hao | `O(n)` |
| 4. BIT | `O(N)` | `O(log N)` | `O(N)` |

In [14]:
def bench(cls, n, ks):
    obj = cls(n); t = time.perf_counter()
    for k in ks:
        obj.fetch(k)
    return (time.perf_counter() - t) * 1000

rng = random.Random(1)
n = 2000; ks = [rng.randint(1, n) for _ in range(2000)]
for cls in (MRUQueueBrute, MRUQueueList, MRUQueueSqrt, MRUQueueBIT, MRUQueue):
    print(f"{cls.__name__:<14} {bench(cls, n, ks):7.1f} ms cho 2000 lần fetch")

MRUQueueBrute     25.8 ms cho 2000 lần fetch
MRUQueueList       0.3 ms cho 2000 lần fetch
MRUQueueSqrt       3.8 ms cho 2000 lần fetch
MRUQueueBIT        4.1 ms cho 2000 lần fetch
MRUQueue           4.1 ms cho 2000 lần fetch


Với `n = 2000`, `list.pop` (Cách 2) thường **nhanh nhất** trong Python vì dịch bộ nhớ bằng `memmove` trong C; BIT có ưu thế tiệm cận nhưng hằng số Python lớn. Đó là bài học thực tế: độ phức tạp tốt hơn chưa chắc nhanh hơn ở kích thước nhỏ.

## 12. Bộ test

Chuỗi lệnh ngẫu nhiên (seed cố định) đúng ràng buộc, đối chiếu mọi cài đặt với Cách 1; thêm edge case `n = 1`, luôn `fetch(n)`, luôn `fetch(1)`.

In [15]:
IMPLS = [MRUQueueList, MRUQueueSqrt, MRUQueueBIT, MRUQueue]
def check_seq(n, ks):
    ops = ["MRUQueue"] + ["fetch"] * len(ks); args = [[n]] + [[k] for k in ks]
    ref = run(MRUQueueBrute, ops, args)
    for cls in IMPLS:
        assert run(cls, ops, args) == ref, (cls.__name__, n, ks)
    return ref

assert check_seq(8, [3, 5, 2, 8]) == ex_out
assert check_seq(1, [1, 1, 1]) == [None, 1, 1, 1]
assert check_seq(5, [5] * 6) == [None, 5, 5, 5, 5, 5, 5]
assert check_seq(4, [1] * 6) == [None, 1, 2, 3, 4, 1, 2]
rng = random.Random(1756)
for _ in range(300):
    n = rng.randint(1, 60)
    check_seq(n, [rng.randint(1, n) for _ in range(rng.randint(1, 80))])
for _ in range(3):
    n = rng.randint(1500, 2000)
    check_seq(n, [rng.randint(1, n) for _ in range(2000)])
print("Edge case + 300 chuỗi nhỏ + 3 chuỗi lớn (2000 lệnh): OK")

Edge case + 300 chuỗi nhỏ + 3 chuỗi lớn (2000 lệnh): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đổi chỉ số 1-based

In [16]:
class WrongIndex:
    def __init__(self, n): self.q = list(range(1, n + 1))
    def fetch(self, k):
        x = self.q.pop(k) if k < len(self.q) else self.q.pop()    # SAI: phải là k - 1
        self.q.append(x); return x

print("sai :", run(WrongIndex, ex_ops, ex_args), "| đúng:", ex_out)

sai : [None, 4, 7, 3, 3] | đúng: [None, 3, 6, 2, 2]


**Vì sao sai:** `fetch(3)` phải lấy phần tử **thứ 3** = `q[2]`. **Sửa:** `pop(k - 1)`.

### Lỗi 2 — Trả phần tử ở vị trí `k` **sau khi** đã di chuyển

In [17]:
class WrongReturn:
    def __init__(self, n): self.q = list(range(1, n + 1))
    def fetch(self, k):
        x = self.q.pop(k - 1); self.q.append(x)
        return self.q[k - 1]                                       # SAI: đọc lại sau khi dịch

print("sai :", run(WrongReturn, ex_ops, ex_args), "| đúng:", ex_out)

sai : [None, 4, 7, 4, 2] | đúng: [None, 3, 6, 2, 2]


**Vì sao sai:** sau `pop`, vị trí `k − 1` đã là phần tử khác. **Sửa:** lưu `x` rồi trả `x`.

### Lỗi 3 — BIT: cấp phát thiếu ô cho các lần `fetch`

In [18]:
class WrongSize(MRUQueueBIT):
    MAX_FETCH = 0                                   # SAI: không chừa ô cho phần tử được chuyển về cuối

try:
    r = run(WrongSize, ex_ops, ex_args)
    print("kết quả sai:", r)
except IndexError as e:
    print("Bắt được IndexError:", e)
print("đúng:", run(MRUQueue, ex_ops, ex_args))

Bắt được IndexError: list assignment index out of range
đúng: [None, 3, 6, 2, 2]


**Vì sao sai:** mỗi `fetch` dùng một ô **mới** ở cuối; cần `n + số lần fetch` ô. Ở đây ô mới vượt mảng (hoặc BIT không cập nhật được ô ngoài phạm vi nên kết quả sai). **Sửa:** `size = n + 2000`.

### Lỗi 4 — Nhảy nhị phân dùng `<=` thay vì `<`

In [19]:
class WrongKth(MRUQueueBIT):
    def _kth(self, k):
        pos, step = 0, 1 << self.size.bit_length()
        while step:
            nxt = pos + step
            if nxt <= self.size and self.tree[nxt] <= k:             # SAI
                pos = nxt; k -= self.tree[nxt]
            step >>= 1
        return pos + 1

try:
    print("sai :", run(WrongKth, ex_ops, ex_args), "| đúng:", ex_out)
except IndexError as e:
    print("Bản sai nhảy quá mảng -> IndexError:", e, "| đúng:", ex_out)
small = WrongKth(8)
print("ô thứ 3 theo bản sai:", small._kth(3), "| theo bản đúng:", MRUQueueBIT(8)._kth(3))

Bản sai nhảy quá mảng -> IndexError: list index out of range | đúng: [None, 3, 6, 2, 2]
ô thứ 3 theo bản sai: 4 | theo bản đúng: 3


**Vì sao sai:** cần vị trí **nhỏ nhất** có tổng tiền tố `≥ k`; với `<=` ta nhảy qua cả vị trí đạt đúng `k` và dừng ở ô sau — ô đó có thể đang tắt (ở đây ô 4 thay vì ô 3), thậm chí vượt khỏi mảng. **Sửa:** điều kiện `tree[nxt] < k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lấy phần tử thứ k + xóa + chèn cuối" lặp nhiều lần ⇒ **order-statistics**: BIT + tìm thứ k, cây phân đoạn, hoặc chia căn.
- Phần tử luôn được chèn **cuối** ⇒ mẹo **ô theo thời gian** (timestamp) biến chèn thành bật một ô mới.
- Bài design ⇒ luôn viết **bộ chạy lệnh** + bản ngây thơ để đối chiếu.
- Ràng buộc nhỏ (`2000`) ⇒ bản `O(n)` đã qua; follow-up mới cần cấu trúc nâng cao.

## 15. Biến thể của bài

1. **Chèn về đầu** thay vì cuối: dùng ô thời gian **giảm dần** (cấp sẵn ô âm/offset ở đầu mảng).
2. **Không giới hạn số lần fetch**: khi hết ô, **nén lại** (dựng lại BIT với `n` ô) — chi phí khấu hao nhỏ.
3. **146 LRU Cache**: phần tử dùng gần nhất về cuối + truy cập theo **khóa** (dict + danh sách liên kết đôi).
4. **1172 Dinner Plate Stacks**, **295/480** order statistics; **315 Count of Smaller Numbers After Self** (BIT).

## 16. So sánh các cách làm

| Cách | Ý tưởng | `fetch` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng lại list | Cắt ghép list mới | `O(n)` | `O(n)` | Đối chiếu |
| 2. `pop` + `append` | Sửa tại chỗ | `O(n)` (nhanh trong C) | `O(n)` | **Đủ qua đề**, phỏng vấn mở đầu |
| 3. Chia căn | Khối `√n` | `O(√n)` | `O(n)` | Follow-up, dễ cài hơn BIT |
| 4. BIT + ô thời gian | Tìm ô bật thứ k | `O(log N)` | `O(N)` | **Follow-up tối ưu** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `MRUQueue(5)` rồi `fetch(1), fetch(1), fetch(5), fetch(3)`.

**Bài 2 (tính tay):** Với BIT `n = 4`, sau `fetch(2)` và `fetch(1)`, những ô nào đang bật và `val` của chúng là gì?

**Bài 3 (điền TODO):** Hoàn thiện `kth_naive(bits, k)` trả vị trí (1-based) của số `1` thứ `k` trong list `bits` (bản tuyến tính, để đối chiếu nhảy nhị phân).

**Bài 4 (tự viết lại):** Viết lớp `MRUQueueFront` trong đó `fetch(k)` chuyển phần tử thứ `k` về **đầu** hàng đợi; dùng BIT với ô thời gian giảm dần.

**Bài 5 (mini challenge):** Viết `MRUQueueCompact` dùng BIT nhưng **không** biết trước số lần `fetch`: khi hết ô thì nén lại toàn bộ (dựng BIT mới kích thước `2n`).

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def kth_naive(bits, k):
    # TODO: duyệt bits, đếm số 1; khi đếm đủ k thì trả chỉ số (1-based)
    pass

In [21]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class MRUQueueFront:
    def __init__(self, n):
        # TODO: ô [Q+1 .. Q+n] chứa 1..n; ô mới cho fetch thứ t là Q+1-t
        pass
    def fetch(self, k):
        pass

class MRUQueueCompact:
    def __init__(self, n):
        # TODO: size = 2n; khi used == size thì dựng lại từ thứ tự hiện tại
        pass
    def fetch(self, k):
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: mô phỏng list. Mức 2: `[1..5]` → `fetch(1)` = 1 → `[2,3,4,5,1]`. Mức 3: `fetch(1)` = 2 → `[3,4,5,1,2]`; `fetch(5)` = 2 (đã ở cuối); `fetch(3)` = 5 → `[3,4,1,2,5]` ⇒ kết quả `[None, 1, 2, 2, 5]`.

**Bài 2.** Mức 1: ban đầu ô 1–4 bật. Mức 2: `fetch(2)` tắt ô 2, bật ô 5 (`val = 2`). Mức 3: `fetch(1)` tắt ô 1, bật ô 6 (`val = 1`) ⇒ bật: ô 3, 4, 5, 6 với val 3, 4, 2, 1.

**Bài 3.** Mức 1: biến đếm `c`. Mức 2: `if b: c += 1`. Mức 3: `if c == k: return i + 1`.

**Bài 4.** Mức 1: cấp `size = Q + n`, phần tử `i` ở ô `Q + i`. Mức 2: lần `fetch` thứ `t` dùng ô `Q + 1 − t` (nhỏ hơn mọi ô hiện có). Mức 3: phần còn lại giống Cách 4.

**Bài 5.** Mức 1: lưu `used`. Mức 2: khi `used == size`, đọc hàng đợi hiện tại (theo ô bật), dựng lại với `size = 2n`. Mức 3: mỗi lần nén `O(n)`, xảy ra sau ít nhất `n` lần fetch ⇒ khấu hao `O(1)`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
b1 = run(MRUQueue, ["MRUQueue", "fetch", "fetch", "fetch", "fetch"], [[5], [1], [1], [5], [3]])
print("Bài 1:", b1); assert b1 == [None, 1, 2, 2, 5]
q = MRUQueueBIT(4); q.fetch(2); q.fetch(1)
on = [s for s in range(1, q.size + 1) if TraceBIT._prefix(q, s) - TraceBIT._prefix(q, s - 1)]
print("Bài 2: ô bật", on, "val", [q.val[s] for s in on]); assert on == [3, 4, 5, 6] and [q.val[s] for s in on] == [3, 4, 2, 1]

def kth_naive(bits, k):
    c = 0
    for i, b in enumerate(bits):
        if b:
            c += 1
            if c == k:
                return i + 1
rng = random.Random(3)
for _ in range(300):
    bits = [rng.randint(0, 1) for _ in range(rng.randint(1, 40))]
    if sum(bits) == 0: continue
    fw = Fenwick(len(bits))
    for i, b in enumerate(bits, 1):
        if b: fw.add(i, 1)
    k = rng.randint(1, sum(bits))
    assert kth(fw, k) == kth_naive(bits, k)
print("Bài 3: OK")

Bài 1: [None, 1, 2, 2, 5]
Bài 2: ô bật [3, 4, 5, 6] val [3, 4, 2, 1]
Bài 3: OK


In [23]:
# Bài 4
class MRUQueueFront:
    Q = 2000
    def __init__(self, n):
        self.size = self.Q + n
        self.bit = Fenwick(self.size); self.val = [0] * (self.size + 1)
        for i in range(1, n + 1):
            self.val[self.Q + i] = i; self.bit.add(self.Q + i, 1)
        self.nxt = self.Q
    def fetch(self, k):
        s = kth(self.bit, k); x = self.val[s]
        self.bit.add(s, -1); self.bit.add(self.nxt, 1); self.val[self.nxt] = x; self.nxt -= 1
        return x

class FrontBrute:
    def __init__(self, n): self.q = list(range(1, n + 1))
    def fetch(self, k):
        x = self.q.pop(k - 1); self.q.insert(0, x); return x

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 30); ks = [rng.randint(1, n) for _ in range(rng.randint(1, 60))]
    ops = ["MRUQueue"] + ["fetch"] * len(ks); args = [[n]] + [[k] for k in ks]
    assert run(MRUQueueFront, ops, args) == run(FrontBrute, ops, args)
print("Bài 4: OK")

Bài 4: OK


In [24]:
# Bài 5
class MRUQueueCompact:
    def __init__(self, n):
        self.n = n; self._build(list(range(1, n + 1)))
    def _build(self, items):
        self.size = 2 * self.n
        self.bit = Fenwick(self.size); self.val = [0] * (self.size + 1)
        for i, v in enumerate(items, 1):
            self.val[i] = v; self.bit.add(i, 1)
        self.used = len(items); self.rebuilds = getattr(self, "rebuilds", -1) + 1
    def fetch(self, k):
        if self.used == self.size:
            self._build([self.val[kth(self.bit, i)] for i in range(1, self.n + 1)])
        s = kth(self.bit, k); x = self.val[s]
        self.bit.add(s, -1); self.used += 1; self.bit.add(self.used, 1); self.val[self.used] = x
        return x

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 20); ks = [rng.randint(1, n) for _ in range(rng.randint(1, 100))]
    ops = ["MRUQueue"] + ["fetch"] * len(ks); args = [[n]] + [[k] for k in ks]
    assert run(MRUQueueCompact, ops, args) == run(MRUQueueBrute, ops, args)
obj = MRUQueueCompact(10)
for k in [1] * 95: obj.fetch(k)
print("Bài 5: OK | n=10, 95 lần fetch -> số lần nén:", obj.rebuilds)

Bài 5: OK | n=10, 95 lần fetch -> số lần nén: 9


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "ô thời gian + BIT tìm thứ k":**

```python
size = n + MAX_OPS; tree = BIT(size); val = [0]*(size+1)
# khởi tạo: ô i bật, val[i] = i
def fetch(k):
    s = kth(k)                      # nhảy nhị phân: tree[nxt] < k
    x = val[s]; add(s, -1)
    used += 1; add(used, 1); val[used] = x
    return x
```

**Checklist:**
- [ ] Đổi `k` (1-based) sang chỉ số đúng?
- [ ] Lưu giá trị **trước** khi di chuyển, trả đúng giá trị đó?
- [ ] BIT đủ `n + số lần fetch` ô?
- [ ] Nhảy nhị phân dùng `<` (tìm vị trí nhỏ nhất có tổng `≥ k`)?
- [ ] Đã test chuỗi lệnh ngẫu nhiên đối chiếu bản `list` đơn giản, `n = 1`, `fetch(n)` liên tục?